# Endometriosis Federated Learning Framework
## Amazon SageMaker Experiment Notebook

This notebook runs the complete experimental pipeline on SageMaker:
1. Environment setup and verification
2. Integration testing
3. Data loading and preprocessing
4. Federated learning training
5. Centralised baseline
6. Comparative analysis
7. Results persistence to S3

## 1. Environment Setup

In [ ]:
import sys
import os

# Set working directory to project root
PROJECT_ROOT = os.path.dirname(os.path.abspath('.'))
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

# Run SageMaker setup
from sagemaker_setup import setup_sagemaker
setup_info = setup_sagemaker()

print(f"\nSetup complete. GPU available: {setup_info['gpu']['available']}")

## 2. Integration Test

In [ ]:
from experiments.run_quick_test import run_quick_test

test_results = run_quick_test()

passed = sum(1 for v in test_results.values() if v == 'PASSED')
print(f"\nTests passed: {passed}/{len(test_results)}")
assert passed == len(test_results), "Some tests failed! Fix before proceeding."

## 3. Data Loading

In [ ]:
from src.utils.seed import set_seed, set_gpu_config
from src.utils.config import load_config
from src.data.dataset_loader import CombinedDatasetLoader
from src.data.partition import DataPartitioner, split_dataset
from src.data.preprocessing import compute_class_weights

# Initialise
set_seed(42)
set_gpu_config()
config = load_config()

# Load data (use synthetic for demo, replace with real data paths)
USE_SYNTHETIC = True  # Set to False when real data is available

if USE_SYNTHETIC:
    images, labels = CombinedDatasetLoader.create_synthetic_dataset(
        num_samples=500,
        image_size=(224, 224),
        seed=42,
    )
else:
    loader = CombinedDatasetLoader(config)
    images, labels = loader.load_all_data()

print(f"Dataset: {images.shape[0]} samples, shape={images.shape[1:]}")
print(f"Class distribution: {dict(zip(*np.unique(labels, return_counts=True)))}")

# Split
splits = split_dataset(images, labels, seed=42)
train_images, train_labels = splits['train']
val_images, val_labels = splits['val']
test_images, test_labels = splits['test']

print(f"\nTrain: {len(train_labels)}, Val: {len(val_labels)}, Test: {len(test_labels)}")

## 4. Federated Learning Training

In [ ]:
from experiments.run_federated import run_single_experiment
from src.utils.logger import get_experiment_logger

logger = get_experiment_logger('sagemaker_federated')

# Run federated experiment
fed_results = run_single_experiment(
    config=config,
    architecture='ResNet50V2',
    epsilon=1.0,
    non_iid=True,
    use_synthetic=USE_SYNTHETIC,
    output_dir='results/federated',
    logger=logger,
)

print(f"\nFederated Results:")
print(f"  Accuracy: {fed_results.get('test_results', {}).get('accuracy', 0):.4f}")
print(f"  AUC: {fed_results.get('test_results', {}).get('roc_auc', 0):.4f}")
print(f"  Convergence round: {fed_results.get('convergence_round', 'N/A')}")

## 5. Centralised Baseline

In [ ]:
from experiments.run_centralised import run_centralised_experiment

logger = get_experiment_logger('sagemaker_centralised')

# Run centralised baseline
cent_results = run_centralised_experiment(
    config=config,
    architecture='ResNet50V2',
    use_synthetic=USE_SYNTHETIC,
    output_dir='results/centralised',
    logger=logger,
)

print(f"\nCentralised Baseline Results:")
print(f"  Accuracy: {cent_results.get('test_results', {}).get('accuracy', 0):.4f}")
print(f"  AUC: {cent_results.get('test_results', {}).get('roc_auc', 0):.4f}")

## 6. Comparative Analysis

In [ ]:
import numpy as np
from src.evaluation.comparison import (
    compare_federated_vs_centralised,
    evaluate_privacy_performance_tradeoff,
)
from src.evaluation.visualisation import (
    plot_convergence_curves,
    plot_privacy_tradeoff,
    plot_confusion_matrix,
)

# Compare federated vs centralised
comparison = compare_federated_vs_centralised(
    federated_results=fed_results.get('test_results', {}),
    centralised_results=cent_results.get('test_results', {}),
)

print("\nFederated vs Centralised Comparison:")
print(f"  {comparison.get('summary', '')}")

for metric, values in comparison.get('metrics', {}).items():
    print(f"  {metric}: Fed={values['federated']:.4f}, Cent={values['centralised']:.4f}, "
          f"Diff={values['absolute_difference']:.4f}")

# Plot confusion matrix if available
if 'confusion_matrix' in fed_results.get('test_results', {}):
    cm = np.array(fed_results['test_results']['confusion_matrix'])
    plot_confusion_matrix(cm, save_path='results/figures/confusion_matrix.png')
    print("\nConfusion matrix saved.")

## 7. Save Results to S3

In [ ]:
from sagemaker_setup import upload_results_to_s3

if setup_info.get('s3_bucket'):
    upload_results_to_s3(
        local_results_path='results',
        bucket_name=setup_info['s3_bucket'],
        s3_prefix='experiment_results',
    )
    print(f"Results uploaded to s3://{setup_info['s3_bucket']}/experiment_results")
else:
    print("No S3 bucket configured. Results saved locally in ./results/")

print("\nExperiment complete!")